In [ ]:
import os, time
import pandas as pd
import geopandas as gpd
import requests
from tqdm import tqdm
from dotenv import load_dotenv
from helpers import SESSION, get_sh_token

load_dotenv()

# Notebook: Compute Viewing Zenith Angles per Window

This notebook processes each training window to compute Sentinel-2 viewing zenith angles (VZA) and data mask statistics using the Copernicus Data Space Ecosystem Statistics API. For each window, it selects the best matching S2 product based on angle similarity and temporal proximity, ensuring perfect data coverage.

Purpose: Generate angle-matched S2-VHR pairs for super-resolution training with geometric consistency.

In [ ]:
WINDOWS_GPKG = "gpkg/windows_2560m_by_datastrip_from_candidates.gpkg"

CAND_PARQUET = "parquet/s2_vhr_candidates_10deg.parquet"
OUT_PARQUET  = "parquet/windows_best_s2_with_vza.parquet"

CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")

# Stats settings
RES_METERS = 60          # 20 or 60 are both fine; 60 is faster
TIME_PAD_MIN = 240       # will be floored to day anyway
SLEEP_S = 0.10           # pacing to reduce throttling

## Evalscript Definition

Define the Sentinel Hub evalscript to extract viewing zenith angles and data mask.

In [3]:
# ---------------------------
# Evalscript: VZA + dataMask
# ---------------------------
EVALSCRIPT_VZA_MASK = """//VERSION=3
function setup() {
  return {
    input: [{ bands: ["viewZenithMean", "dataMask"] }],
    output: [
      { id: "vza", bands: 1, sampleType: "FLOAT32" },
      { id: "dataMask", bands: 1, sampleType: "UINT8" }
    ]
  };
}
function evaluatePixel(s) {
  return {
    vza: [s.viewZenithMean],
    dataMask: [s.dataMask]
  };
}
"""

## Statistics Function

Function to query viewing zenith angles and data mask statistics for a given window and S2 time.

In [ ]:
def stats_vza_and_mask(bbox_3035, s2_time_iso):
    """Return dict with vza stats + mask stats for this bbox on the UTC day of s2_time."""
    # ensure JSON-serializable bbox
    bbox = [float(x) for x in bbox_3035]

    t = pd.to_datetime(s2_time_iso, utc=True)
    day_start = t.normalize()
    day_end   = day_start + pd.Timedelta(days=1)

    day_from = day_start.strftime("%Y-%m-%dT%H:%M:%SZ")
    day_to   = day_end.strftime("%Y-%m-%dT%H:%M:%SZ")

    payload = {
        "input": {
            "bounds": {
                "bbox": bbox,
                "properties": {"crs": "http://www.opengis.net/def/crs/EPSG/0/3035"},
            },
            "data": [{
                "type": "sentinel-2-l2a",
                "dataFilter": {"mosaickingOrder": "mostRecent"},
            }],
        },
        "aggregation": {
            "timeRange": {"from": day_from, "to": day_to},
            "aggregationInterval": {"of": "P1D"},
            "evalscript": EVALSCRIPT_VZA_MASK,
            "resx": RES_METERS,
            "resy": RES_METERS,
        },
    }

    for attempt in range(6):
        token = get_sh_token()
        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
            "Accept": "application/json",
        }

        r = SESSION.post(os.getenv("STATS_URL"), headers=headers, json=payload, timeout=180)

        if r.status_code == 200:
            js = r.json()
            # Can be OK but empty
            if not js.get("data"):
                return None
            d0 = js["data"][0]["outputs"]

            vza_stats = d0["vza"]["bands"]["B0"]["stats"]
            return {
                "vza_min": float(vza_stats["min"]),
                "vza_mean": float(vza_stats["mean"]),
                "vza_max": float(vza_stats["max"]),
                "mask_sampleCount": int(vza_stats["sampleCount"]),
                "mask_noDataCount": int(vza_stats["noDataCount"]),
            }

        # expired token -> retry
        if r.status_code == 401:
            time.sleep(1)
            continue

        # throttling/transient
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(2 * (attempt + 1))
            continue

        # hard failure
        raise requests.HTTPError(f"{r.status_code} {r.text[:1000]}", response=r)

    raise RuntimeError("Stats API failed after retries")


## Load Data

Load window geometries and candidate S2 products, preparing for processing.

In [7]:
win = gpd.read_file(WINDOWS_GPKG)
cand = pd.read_parquet(CAND_PARQUET)

# Optional: only L2A names if your candidates include L1C
cand = cand[cand["s2_name"].str.contains("_MSIL2A", na=False)].copy()

# We'll lookup candidates by datastrip quickly
cand_by_ds = {ds: df for ds, df in cand.groupby("datastrip")}

results = []
stats_cache = {}  # cache by (bbox, s2_day) to avoid repeated calls for same window/s2

## Main Processing Loop

Iterate through windows, query S2 statistics, and select the best matching S2 product for each window.

In [8]:
# ---------------------------
# Main loop over windows
# ---------------------------
for _, w in tqdm(win.iterrows(), total=len(win), desc="Window VZA+mask and choose best S2"):
    ds = w["datastrip"]
    if ds not in cand_by_ds:
        continue

    bbox = [w["minx"], w["miny"], w["maxx"], w["maxy"]]

    # candidate S2 rows for this datastrip
    cands = cand_by_ds[ds]

    best = None
    best_key = None

    for _, c in cands.iterrows():
        s2_time = c["s2_time"]
        vhr_time = c["vhr_time"]
        ona = float(c["ona_angle"])

        # cache key (window bbox + s2 day)
        t = pd.to_datetime(s2_time, utc=True)
        s2_day = t.floor("D").strftime("%Y-%m-%d")
        cache_key = (tuple(map(float, bbox)), s2_day)

        if cache_key in stats_cache:
            st = stats_cache[cache_key]
        else:
            st = stats_vza_and_mask(bbox, s2_time)
            stats_cache[cache_key] = st
            time.sleep(SLEEP_S)

        if st is None:
            continue

        # Hard filter: require perfect coverage in the window
        if st["mask_noDataCount"] != 0:
            continue

        # Rank: smallest abs(vza_mean - ona_angle), then smallest time diff
        vza_mean = st["vza_mean"]
        angle_abs = abs(vza_mean - ona)
        dt_abs = abs((pd.to_datetime(s2_time, utc=True) - pd.to_datetime(vhr_time, utc=True)).total_seconds())

        score = (angle_abs, dt_abs)

        if best is None or score < best_key:
            best_key = score
            best = {
                "window_id": w["window_id"],
                "datastrip": ds,
                "geometry_wkt": w.geometry.wkt,  # store WKT in parquet; or write a new gpkg later
                "minx": float(bbox[0]), "miny": float(bbox[1]), "maxx": float(bbox[2]), "maxy": float(bbox[3]),

                "vhr_time": c["vhr_time"],
                "ona_angle": ona,

                "s2_id": c["s2_id"],
                "s2_name": c["s2_name"],
                "s2_time": c["s2_time"],

                "vza_min": st["vza_min"],
                "vza_mean": st["vza_mean"],
                "vza_max": st["vza_max"],
                "mask_sampleCount": st["mask_sampleCount"],
                "mask_noDataCount": st["mask_noDataCount"],

                # signed diffs (keep sign!)
                "angle_diff_mean": st["vza_mean"] - ona,
                "angle_diff_min":  st["vza_min"]  - ona,
                "angle_diff_max":  st["vza_max"]  - ona,
            }

    if best is not None:
        results.append(best)

# Save
df_out = pd.DataFrame(results)
df_out.to_parquet(OUT_PARQUET, index=False)
print(f"Saved: {OUT_PARQUET}")
print("Kept windows:", len(df_out), "/", len(win))

Window VZA+mask and choose best S2: 100%|██████████| 74821/74821 [10:15:33<00:00,  2.03it/s]  


Saved: windows_best_s2_with_vza.parquet
Kept windows: 66747 / 74821
